# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityag30/FlyRank-internship-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*



The feature vector uses observable search and engagement signals aggregated for the March 2026 warehouse snapshot. Two engineered features, click-through rate (CTR) and engagement rate, are calculated from the observed metrics. Identifier fields and target-derived variables are intentionally excluded to reduce leakage risk.

In [3]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

In [6]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [8]:
query = f"""
DESCRIBE SELECT *
FROM {TABLES['fact_daily']}
"""

con.sql(query).df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [10]:
import pandas as pd

# Build a small feature frame from the March 2026 warehouse slice
query = f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions,
    CASE
        WHEN gsc_impressions > 0
        THEN gsc_clicks * 1.0 / gsc_impressions
        ELSE 0
    END AS ctr,
    CASE
        WHEN ga4_sessions > 0
        THEN ga4_engaged_sessions * 1.0 / ga4_sessions
        ELSE 0
    END AS engagement_rate
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

X = con.sql(query).df()

# Fill missing values
X = X.fillna(0)

print("Feature matrix shape:", X.shape)
X.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature matrix shape: (9841378, 7)


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,ctr,engagement_rate
0,20,0,3.350000,0,0,0.000,0.0
1,1,0,0.000000,0,0,0.000,0.0
2,125,1,4.928000,0,0,0.008,0.0
3,7,0,4.000000,0,0,0.000,0.0
4,11,0,2.272727,0,0,0.000,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


| Feature | Meaning | Missing value handling | Available before prediction? |
|---------|---------|------------------------|------------------------------|
| gsc_impressions | Number of Google Search impressions | Filled with 0 | Yes |
| gsc_clicks | Number of Google Search clicks | Filled with 0 | Yes |
| gsc_avg_position | Average Google Search ranking position | Filled with median (or 0 if unavailable) | Yes |
| ga4_sessions | Website sessions from Google Analytics | Filled with 0 | Yes |
| ga4_engaged_sessions | Engaged sessions from Google Analytics | Filled with 0 | Yes |
| ctr | Click-through rate (clicks ÷ impressions) | Set to 0 when impressions = 0 | Yes |
| engagement_rate | Engaged sessions ÷ sessions | Set to 0 when sessions = 0 | Yes |

No categorical variables are used in this feature vector. All selected features are available before making the prediction, reducing the risk of target leakage.

In [11]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

feature_notes = pd.DataFrame({
    "Feature": [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "ctr",
        "engagement_rate"
    ],
    "Missing Handling": [
        "Fill 0",
        "Fill 0",
        "Median / 0",
        "Fill 0",
        "Fill 0",
        "0 if impressions = 0",
        "0 if sessions = 0"
    ],
    "Categorical": [
        "No",
        "No",
        "No",
        "No",
        "No",
        "No",
        "No"
    ],
    "Available Before Prediction": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

feature_notes

,Feature,Missing Handling,Categorical,Available Before Prediction
0,gsc_impressions,Fill 0,No,Yes
1,gsc_clicks,Fill 0,No,Yes
2,gsc_avg_position,Median / 0,No,Yes
3,ga4_sessions,Fill 0,No,Yes
4,ga4_engaged_sessions,Fill 0,No,Yes
5,ctr,0 if impressions = 0,No,Yes
6,engagement_rate,0 if sessions = 0,No,Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*



The feature vector was reviewed for common sources of leakage.

- No label-derived columns were included.
- No future-window metrics were used.
- No post-decision product flags or outcome variables were used.
- All features are available at the time a prediction would be made.

The engineered features (CTR and engagement rate) are computed only from contemporaneous observations and do not use future information.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Candidate features used for the model
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ctr",
    "engagement_rate",
]

# Common leakage keywords
leakage_keywords = [
    "label",
    "target",
    "future",
    "outcome",
    "converted",
    "conversion",
    "refresh",
    "flag",
    "action",
    "next",
    "after"
]

# Check feature names for suspicious columns
suspect = [
    col for col in feature_cols
    if any(word in col.lower() for word in leakage_keywords)
]

print("Features:", feature_cols)

if suspect:
    print("⚠ Potential leakage columns found:", suspect)
else:
    print("✓ No obvious leakage columns detected.")

Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'ctr', 'engagement_rate']
✓ No obvious leakage columns detected.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*



The following fields were intentionally excluded from the feature vector because they introduce leakage, identify records rather than describe them, or are only available after the prediction decision.

| Field | Reason for Exclusion |
|-------|----------------------|
| final_rank | Final ranking produced after scoring; target-related information. |
| final_refresh_score | Final model score; directly leaks the prediction target. |
| best_model_name | Generated after model evaluation; unavailable before prediction. |
| best_model_probability | Model output probability; direct leakage. |
| baseline_refresh_score | Baseline model prediction; leaks target information. |
| confidence | Computed after prediction; unavailable beforehand. |
| suggested_action | Recommendation generated after scoring. |
| final_reason_codes | Explanation generated after prediction. |
| is_declining_label | Target label used for training and evaluation. |
| content_id | Unique identifier with no predictive value. |
| client_id | Identifier that may cause memorization rather than generalization. |

In [13]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

excluded_fields = pd.DataFrame({
    "Field": [
        "final_rank",
        "final_refresh_score",
        "best_model_name",
        "best_model_probability",
        "baseline_refresh_score",
        "confidence",
        "suggested_action",
        "final_reason_codes",
        "is_declining_label",
        "content_id",
        "client_id"
    ],
    "Reason": [
        "Final ranking after scoring",
        "Model prediction output",
        "Selected after model evaluation",
        "Model probability output",
        "Baseline prediction output",
        "Computed after prediction",
        "Post-prediction recommendation",
        "Explanation generated after prediction",
        "Training target",
        "Identifier only",
        "Identifier only"
    ]
})

excluded_fields

,Field,Reason
0,final_rank,Final ranking after scoring
1,final_refresh_score,Model prediction output
2,best_model_name,Selected after model evaluation
3,best_model_probability,Model probability output
4,baseline_refresh_score,Baseline prediction output
5,confidence,Computed after prediction
6,suggested_action,Post-prediction recommendation
7,final_reason_codes,Explanation generated after prediction
8,is_declining_label,Training target
9,content_id,Identifier only


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.